In [ ]:
%reset -f
# 导入必要的库
import pyvisa
import os
import struct
import time
import function
from imp import reload
import function 
from importlib import reload
reload(function)
from RsSmw import *
import numpy as np
import matplotlib.pyplot as plt

# ============================
# 分析模式控制开关
# ============================
ENABLE_ILC_ITERATIONS = 1      # 是否执行ILC迭代循环
ENABLE_COMPARISON_PLOTS = 1     # 是否绘制对比分析图
ENABLE_SMW200A = 1              # 是否启用信号发生器
ENABLE_SWEEP = 1                # 是否执行扫频分析
ENABLE_ACPR = 1                 # 是否执行邻道功率测量
ENABLE_IQ_ANALYSIS = 1          # 是否执行IQ分析
ENABLE_SAVE_IQ = 1              # 是否保存IQ数据到文件
ENABLE_PLOT_PSD = 1             # 是否绘制功率谱密度图
ENABLE_UPLOAD_WAVEFORM = 1      # 是否上传波形文件到信号发生器

# ============================
# 全局参数设置
# ============================

# 注意：控制开关已在开头统一定义

# 设备IP地址
SMW200A_IP = "192.168.1.201"
CEYEAR_IP = "192.168.1.202"

# 频率和功率设置
CENTER_FREQ = "3.5GHz"                     # 中心频率
POWER_LEVEL = "-12"                        # 功率电平(dBm)
SAMPLING_RATE = "491.52MHz"                # 采样率
SAMPLING_RATE_HZ = 491.52e6                # 采样率(Hz)
# MEASUREMENT_TIME = "0.001"                 # 测量时间(秒)
# IQ_RECORD_LENGTH = "491520"                # IQ记录长度
ATT_LEVEL = "13dB"
Y_SCAL = "9dBm"
IQ_RECORD_LENGTH = "100000"                # IQ记录长度

SWEEP_RANGE = "500MHz"                     # 扫频分析中扫频带宽
RBW_OCCUPIED = "1MHz"                      # 占用带宽测量RBW
VBW_OCCUPIED = "3MHz"                      # 占用带宽测量VBW

# 频谱仪RBW/VBW设置 (基于5G NR 100MHz信号)
RBW_ACLR = "100kHz"                        # ACLR测量RBW (满足3GPP要求)
VBW_ACLR = "300kHz"                        # ACLR测量VBW

# 扫频分析设置
SWEEP_RANGE = "500MHz"                     # 扫频范围
RBW_OCCUPIED = "1MHz"                      # 占用带宽测量RBW
VBW_OCCUPIED = "3MHz"                      # 占用带宽测量VBW

# ACP测量参数
CARRIER_BW = "98.28MHz"                      # 载波带宽
ADJACENT_CH_OFFSET = "100MHz, 200MHz"        # 邻道偏移
ADJACENT_CH_BW = CARRIER_BW + ',' + CARRIER_BW  # 邻道带宽

# ILC算法参数
ILC_ITERATIONS = 4                         # ILC迭代次数
only_ILC_ITERATIONS = 3                 # 仅进行第only_ILC_ITERATIONS次ILC迭代
ILC_GAMMA = 1.0                           # ILC学习增益 (0.5-1.5)

# 文件路径和名称
date = time.strftime("%Y%m%d", time.localtime()) + "/"
LOCAL_DIR = f"C:\\Users\\25834\\Desktop\\PA\\data\\3.5GHz\\{date.replace('/', '')}\\"
data_ini_path = "C:/Users/25834/Desktop/PA/data/"
data_path = "C:/Users/25834/Desktop/PA/data/3.5GHz/"
data_out_path = data_path + date

# 信号文件名
data_ini_name = "NR-FR1-TM3.1a_100MHz_491.52Msps_30kHz_FDD_8.113dB"

# R&S SMW200A波形文件设置
INST_PRESET_WV_FILE = "USER/zwh/NR-FR1-TM3.1a_100MHz_491.52Msps_30kHz_FDD_8.113dB.wv"
INST_WV_DIR = f"USER/zwh/{date.replace('/', '')}"

# 创建本地数据保存目录
os.makedirs(LOCAL_DIR, exist_ok=True)
os.makedirs(data_out_path, exist_ok=True)

print(f"数据保存路径: {LOCAL_DIR}")
print(f"迭代次数: {ILC_ITERATIONS}")

# ============================
# 载入原始信号数据
# ============================
print("\\n载入原始信号数据...")
data_ini_IQ, _, _ = load_data(data_ini_path, data_ini_name)
initial_signal = data_ini_IQ[:, 0] + 1j*data_ini_IQ[:, 1]
print(f"原始信号载入完成，长度: {len(initial_signal)}")

# ============================
# ILC迭代循环处理
# ============================

if ENABLE_ILC_ITERATIONS:
    print("\\n=== 开始ILC迭代循环处理 ===")
    print(f"学习增益 gamma = {ILC_GAMMA}")
    
    # 存储每次迭代的结果
    aclr_results_history = []
    pa_output_history = []
    pa_input_history = []
    nmse_history = []
    papr_history = []
    
    current_signal = initial_signal  # 当前信号发生器要发射的信号    
    for iteration in range(ILC_ITERATIONS + 1):  # +1 是因为第0次是原始信号

    # for iteration in range(only_ILC_ITERATIONS, only_ILC_ITERATIONS + 1):
    #     _, _, current_signal = load_data(LOCAL_DIR, f'ilc_signal_iter_{iteration}.txt')
        print(f"\\n--- 第 {iteration} 次迭代 ---")
        
        # ============================
        # 步骤1：配置信号发生器发射当前信号
        # ============================
        if ENABLE_SMW200A:
            if iteration == 0:
                print("信号发生器发射原始信号...")
                # 第0次：发射预设的原始信号
                rm_SMW200 = pyvisa.ResourceManager()
                usr_SMW200 = rm_SMW200.open_resource(f"TCPIP0::{SMW200A_IP}::inst0::INSTR")
                usr_SMW200.set_visa_attribute(pyvisa.constants.VI_ATTR_TERMCHAR_EN, pyvisa.constants.VI_TRUE)
                usr_SMW200.set_visa_attribute(pyvisa.constants.VI_ATTR_TERMCHAR, ord('\n'))
                usr_SMW200.timeout = 30000
                
                usr_SMW200.write("OUTPut:STATe OFF")
                usr_SMW200.write(f"FREQ {CENTER_FREQ}")
                usr_SMW200.write(f"POW {POWER_LEVEL}")
                usr_SMW200.write("SOURce1:BB:ARBitrary:TRIGger:OUTPut1:MODE RESTart")
                usr_SMW200.write("OUTPut:USER1:DIRection OUTP")
                usr_SMW200.write("OUTPut:USER1:SIGNal MARKA1")
                usr_SMW200.write("OUTPut:USER4:DIRection OUTP")
                usr_SMW200.write("OUTPut:USER4:SIGNal MARKA1")
                
                usr_SMW200.write(f'SOURce:BB:ARB:WAV:SELect "{INST_PRESET_WV_FILE}"')
                usr_SMW200.write("SOURce:BB:ARB:STATe ON")
                usr_SMW200.write("OUTPut:STATe ON")
                
                usr_SMW200.close()
                rm_SMW200.close()
            else:
                print(f"信号发生器发射第{iteration}次ILC信号...")
                # 第1次及以后：发射ILC处理后的信号
                
                # 创建并上传ILC波形文件
                RsSmw.assert_minimum_version('5.0.44')
                smw = RsSmw(f'TCPIP0::{SMW200A_IP}::inst0::INSTR')
                rm = pyvisa.ResourceManager()
                usr_SMW200 = rm.open_resource(f"TCPIP0::{SMW200A_IP}::inst0::INSTR")
                usr_SMW200.timeout = 30000
                
                usr_SMW200.write('OUTP:STAT OFF')
                usr_SMW200.write(f'MMEM:MDIR "{INST_WV_DIR}"')
                
                # # 保存当前信号到文件
                # current_signal_file = os.path.join(LOCAL_DIR, f'ilc_signal_iter_{iteration}.txt')
                # current_signal_data = np.column_stack([current_signal.real, current_signal.imag])
                # np.savetxt(current_signal_file, current_signal_data, fmt='%.8f')
                
                # 创建波形文件
                LOCAL_WV_FILE = os.path.join(LOCAL_DIR, f'ILC_iter_{iteration}.wv')
                INST_WV_FILE = f"{INST_WV_DIR}/ILC_iter_{iteration}.wv"
                
                smw.arb_files.create_waveform_file_from_samples(
                    current_signal.real, current_signal.imag, LOCAL_WV_FILE,
                    clock_freq=SAMPLING_RATE_HZ, auto_scale=True,
                    additional_tags=['MARKER LIST 1: 0:1;10:0'],
                    comment=f'ILC iteration {iteration}'
                )
                
                # 上传波形文件
                with open(LOCAL_WV_FILE, 'rb') as file:
                    file_content = file.read()
                
                data_len = len(file_content)
                len_str = str(data_len)
                header = f'#{len(len_str)}{len_str}'
                
                usr_SMW200.write_raw(f'MMEM:DATA "{INST_WV_FILE}",'.encode() + header.encode() + file_content)
                usr_SMW200.write(f'SOURce:BB:ARB:WAV:SELect "{INST_WV_FILE}"')
                usr_SMW200.write('SOURce:BB:ARB:STATe ON')
                usr_SMW200.write('OUTP:STAT ON')
                
                usr_SMW200.close()
                rm.close()
        
        # ============================
        # 步骤2：频谱仪测量PA输出
        # ============================
        print("频谱仪测量PA输出...")
        rm_ceyear = pyvisa.ResourceManager()
        usr_ceyear = rm_ceyear.open_resource(f"TCPIP0::{CEYEAR_IP}::5025::SOCKET")
        usr_ceyear.set_visa_attribute(pyvisa.constants.VI_ATTR_TERMCHAR_EN, pyvisa.constants.VI_TRUE)
        usr_ceyear.set_visa_attribute(pyvisa.constants.VI_ATTR_TERMCHAR, ord('\n'))
        usr_ceyear.set_visa_attribute(pyvisa.constants.VI_ATTR_TMO_VALUE, 10000)
        
        # ACLR测量
        if ENABLE_ACPR:
            usr_ceyear.write(":CONFigure:ACPower")
            time.sleep(1)
            usr_ceyear.write(f":FREQ:CENT {CENTER_FREQ}")
            # usr_ceyear.write(f":ACP:BAND {RBW_ACLR}")
            # usr_ceyear.write(f":ACP:BAND:VID {VBW_ACLR}")
            usr_ceyear.write(":RAD:STAN NR5GFR1B100M")
            usr_ceyear.write(":TRAC1:ACP:TYPE AVERage")
            usr_ceyear.write(":FETC:ACP?")
            acp_results = usr_ceyear.read().strip().split(',')
            aclr_results_history.append(acp_results)
            time.sleep(5)
            
            print(f"第{iteration}次ACLR结果:")
            print(f"  载波功率: {acp_results[0]} dBm")
            print(f"  邻道: 低相对: {acp_results[2]} dBc, 高相对: {acp_results[4]} dBc")
            print(f"  次邻道: 低相对: {acp_results[6]} dBc, 高相对: {acp_results[8]} dBc")
                
        # IQ数据采集
        if ENABLE_IQ_ANALYSIS:
            usr_ceyear.write(":INST BASIC")
            usr_ceyear.write(f":FREQ:CENT {CENTER_FREQ}")
            usr_ceyear.write("ROSC:SOUR E10")
            usr_ceyear.write("TRIG:SOUR EXT1")
            usr_ceyear.write(f'TRAC:IQ:SRAT {SAMPLING_RATE}')
            # usr_ceyear.write(f'SWE:TIME {MEASUREMENT_TIME}')
            usr_ceyear.write(f'TRAC:IQ:RLEN {IQ_RECORD_LENGTH}')
            # usr_ceyear.write(':ADJ:LEV')
            usr_ceyear.write(f":INP:ATT {ATT_LEVEL}")                 # 机械衰减器
            usr_ceyear.write(f":DISP:WIND:TRAC:Y:SCAL:RLEV {Y_SCAL}") # 参考电平
            # usr_ceyear.write(f':DISP:WIND:TRAC:Y:SCAL {Y_SCAL}')      # 参考值
            
            time.sleep(5)
            
            usr_ceyear.write('INIT:IMM')
            usr_ceyear.clear()
            time.sleep(1)
            
            # 读取IQ数据
            usr_ceyear.timeout = 30000
            usr_ceyear.write(":FORM:DATA REAL,32")
            usr_ceyear.write(":TRAC:IQ:DATA?")
            
            usr_ceyear.set_visa_attribute(pyvisa.constants.VI_ATTR_TERMCHAR_EN, pyvisa.constants.VI_FALSE)
        
            # 读取二进制块头
            header = usr_ceyear.read_bytes(2)
            digit_count = header[1] - 48
            length_str = usr_ceyear.read_bytes(digit_count).decode('ascii')
            data_length = int(length_str)
            
            # 分块读取数据
            raw_data = bytearray()
            chunk_size = 1024 * 1024
            remaining = data_length
            
            while remaining > 0:
                chunk = min(remaining, chunk_size)
                raw_data.extend(usr_ceyear.read_bytes(chunk))
                remaining -= chunk
            
            usr_ceyear.read_bytes(1)
            
            # 转换为浮点数数组
            float_count = len(raw_data) // 4
            iq_data_current = struct.unpack(f'{float_count}f', raw_data)
            usr_ceyear.set_visa_attribute(pyvisa.constants.VI_ATTR_TERMCHAR_EN, pyvisa.constants.VI_TRUE)
            
            # 转换为复数并保存
            pa_output_current = np.array([[iq_data_current[i], iq_data_current[i+1]] 
                                         for i in range(0, len(iq_data_current), 2)])
            pa_output_complex = pa_output_current[:, 0] + 1j*pa_output_current[:, 1]
            pa_output_history.append(pa_output_complex)
            
            # 保存PA输出数据
            if ENABLE_SAVE_IQ:
                pa_output_file = os.path.join(LOCAL_DIR, f'ILC_output_iter_{iteration}.txt')
                np.savetxt(pa_output_file, pa_output_current, fmt='%.8f')
                print(f"第{iteration}次PA输出数据已保存，共{len(pa_output_complex)}个点")
            else:
                print(f"第{iteration}次PA输出数据已采集，共{len(pa_output_complex)}个点（未保存）")
        
            # 恢复连续模式
            usr_ceyear.write('INIT:CONT ON')


        # 配置扫频分析设置
        if ENABLE_SWEEP:
            print("执行扫频分析...")
            usr_ceyear.write(":INST SA")                     # 切换到频谱分析模式
            usr_ceyear.write(f":FREQ:CENT {CENTER_FREQ}")    # 设置中心频率
            usr_ceyear.write(f":FREQ:SPAN {SWEEP_RANGE}")    # 设置扫频范围
            
            # 设置RBW和VBW用于占用带宽测量
            usr_ceyear.write(f":BAND {RBW_OCCUPIED}")    # 设置分辨带宽
            usr_ceyear.write(f":BAND:VID {VBW_OCCUPIED}")    # 设置视频带宽
            
            time.sleep(1)
            
            # 配置6条曲线的循环显示逻辑
            if iteration == 0:
                # 第0次迭代：测量原始信号PA输出，固定为曲线1（永久参考）
                for i in range(1, 7):
                    usr_ceyear.write(f":TRAC{i}:DISP OFF")
                usr_ceyear.write(":TRAC1:DISP ON")
                usr_ceyear.write(":TRAC1:TYPE AVER")         # 曲线1设置为平均模式
                usr_ceyear.write(":TRAC1:UPD ON")            # 开启曲线1更新
                time.sleep(5)  # 等待平均完成
                usr_ceyear.write(":TRAC1:UPD OFF")           # 固定曲线1（永久参考）
            
            else:
                # 第1次及以后迭代：使用曲线2-6循环
                # 当前曲线号 = 2 + (iteration-1) % 5，范围是2-6
                current_trace = 2 + (iteration - 1) % 5
                
                # # 如果不是第1次迭代，先固定前一条曲线
                # if iteration > 1:
                #     prev_trace = 2 + (iteration - 2) % 5
                    
                #     usr_ceyear.write(f":TRAC{prev_trace}:DISP ON")
                #     usr_ceyear.write(f":TRAC{prev_trace}:UPD OFF")  # 固定前一条曲线
                
                # 设置当前曲线进行测量
                usr_ceyear.write(f":TRAC{current_trace}:DISP ON")
                usr_ceyear.write(f":TRAC{current_trace}:TYPE AVER")  # 当前曲线设置为平均模式
                usr_ceyear.write(f":TRAC{current_trace}:UPD ON")     # 开启曲线1更新
                time.sleep(5)  # 等待前一条曲线平均完成
                usr_ceyear.write(f":TRAC{current_trace}:UPD OFF")    # 固定当前曲线

                
                usr_ceyear.write(f":TRAC{current_trace+1}:DISP ON")
                usr_ceyear.write(f":TRAC{current_trace+1}:TYPE AVER")  # 当前曲线设置为平均模式                
                usr_ceyear.write(f":TRAC{current_trace+1}:UPD ON")
                time.sleep(5)  # 等待前一条曲线平均完成
                
        usr_ceyear.close()
        rm_ceyear.close()
        
        # print(f"第{iteration}次PA输出数据已保存，共{len(pa_output_complex)}个点")
        
        # ============================
        # 步骤3：ILC处理（除了最后一次）
        # ============================
        if iteration < ILC_ITERATIONS:
            print(f"\\n执行第{iteration+1}次ILC处理...")
            
            # 多组数据对齐
            data = [initial_signal, current_signal, pa_output_complex]
            data_in_final, data_out_final = multi_align(data, SAMPLING_RATE_HZ, 25)
            
            print(f"数据对齐完成")
            
            # 归一化
            pa_ini_c = data_in_final / np.max(np.abs(data_in_final))  # 期望输出(原始信号)
            pa_in_c = data_out_final[0] / np.max(np.abs(data_out_final[0]))  # 当前输入
            pa_out_c = data_out_final[1] / np.max(np.abs(data_out_final[1]))  # 实际输出
            
            # 保存归一化信号历史
            pa_input_history.append(pa_in_c)
            
            # 计算PAPR
            PAPR_pa_ini = cal_PAPR(pa_ini_c)
            PAPR_pa_in = cal_PAPR(pa_in_c)
            PAPR_pa_out = cal_PAPR(pa_out_c)
            
            # 计算NMSE
            NMSE_out = cal_NMSE_np(pa_ini_c, pa_out_c)
            nmse_history.append(NMSE_out)
            
            print(f'PAPR - 原始: {PAPR_pa_ini:.4f}dB, 输入: {PAPR_pa_in:.4f}dB, 输出: {PAPR_pa_out:.4f}dB')
            print(f'NMSE: {NMSE_out:.4f}dB')
            
            # ILC算法：计算新的功放输入
            e = pa_ini_c - pa_out_c  # 误差信号
            pa_in_new = pa_in_c + ILC_GAMMA * e  # ILC更新规律
            
            PAPR_pa_new = cal_PAPR(pa_in_new)
            papr_history.append(PAPR_pa_new)
            
            print(f'ILC更新后PAPR: {PAPR_pa_new:.4f}dB')
            
            # 保存ILC信号
            ilc_file_name = f"ILC_iter_{iteration+1}.txt"
            ilc_path = os.path.join(data_out_path, ilc_file_name)
            save_txt_func(pa_in_new, ilc_path)
            print(f"ILC信号已保存到: {ilc_path}")
            
            # 更新当前信号为下一次迭代的输入
            current_signal = pa_in_new
    
    print("\\nILC迭代循环完成！")

else:
    print("已跳过ILC迭代循环")


In [ ]:
# ============================
# ILC效果对比分析
# ============================
ENABLE_COMPARISON = 1
if ENABLE_COMPARISON and ENABLE_ILC_ITERATIONS:
    print("\\n=== ILC效果对比分析 ===")
    
    # ACLR改善趋势分析
    print("\\n=== ACLR迭代改善趋势 ===")
    for i, acp_results in enumerate(aclr_results_history):
        print(f"第{i}次迭代 ACLR:")
        print(f"  载波功率: {acp_results[0]} dBm")
        print(f"  邻道: 低 {acp_results[2]} dBc, 高 {acp_results[4]} dBc")
        print(f"  次邻道: 低 {acp_results[6]} dBc, 高 {acp_results[8]} dBc")
        
        if i > 0:
            # 计算相对于第0次的改善量
            adj_low_improve = float(acp_results[2]) - float(aclr_results_history[0][2])
            adj_high_improve = float(acp_results[4]) - float(aclr_results_history[0][4])
            alt_low_improve = float(acp_results[6]) - float(aclr_results_history[0][6])
            alt_high_improve = float(acp_results[8]) - float(aclr_results_history[0][8])
            
            print(f"  改善量: 邻道 低{adj_low_improve:.2f}dB 高{adj_high_improve:.2f}dB, 次邻道 低{alt_low_improve:.2f}dB 高{alt_high_improve:.2f}dB")
    
    # NMSE改善趋势分析
    print("\\n=== NMSE迭代改善趋势 ===")
    for i, nmse in enumerate(nmse_history):
        print(f"第{i+1}次迭代 NMSE: {nmse:.4f} dB")
        if i > 0:
            nmse_improve = nmse - nmse_history[0]
            print(f"  相对第1次改善: {nmse_improve:.4f} dB")
    
if ENABLE_COMPARISON_PLOTS:
    # 绘制ILC迭代效果对比图
    print("\\n绘制ILC迭代效果对比图...")
    
    # 创建对比图
    fig, axes = plt.subplots(2, 3, figsize=(24, 12))
    
    # ACLR迭代趋势图
    iterations = range(len(aclr_results_history))
    adj_low = [float(results[2]) for results in aclr_results_history]
    adj_high = [float(results[4]) for results in aclr_results_history]
    alt_low = [float(results[6]) for results in aclr_results_history]
    alt_high = [float(results[8]) for results in aclr_results_history]
    
    axes[0, 0].plot(iterations, adj_low, 'bo-', label='邻道低', linewidth=2, markersize=6)
    axes[0, 0].plot(iterations, adj_high, 'ro-', label='邻道高', linewidth=2, markersize=6)
    axes[0, 0].plot(iterations, alt_low, 'go-', label='次邻道低', linewidth=2, markersize=6)
    axes[0, 0].plot(iterations, alt_high, 'mo-', label='次邻道高', linewidth=2, markersize=6)
    axes[0, 0].set_title('ACLR迭代改善趋势', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('迭代次数')
    axes[0, 0].set_ylabel('ACLR (dBc)')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # NMSE迭代趋势图
    nmse_iterations = range(1, len(nmse_history) + 1)
    axes[0, 1].plot(nmse_iterations, nmse_history, 'ko-', label='NMSE', linewidth=2, markersize=6)
    axes[0, 1].set_title('NMSE迭代改善趋势', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('迭代次数')
    axes[0, 1].set_ylabel('NMSE (dB)')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # PAPR迭代趋势图
    papr_iterations = range(1, len(papr_history) + 1)
    axes[0, 2].plot(papr_iterations, papr_history, 'co-', label='PAPR', linewidth=2, markersize=6)
    axes[0, 2].set_title('PAPR迭代变化趋势', fontsize=14, fontweight='bold')
    axes[0, 2].set_xlabel('迭代次数')
    axes[0, 2].set_ylabel('PAPR (dB)')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # AM/AM特性对比（原始 vs 最终ILC）
    # 使用multi_align函数对齐数据
    print("对齐AM/AM图表数据...")
    
    # 对齐初始数据    
    initial_aligned, initial_out_aligned,_,_ = alignment(initial_signal, pa_output_history[0], SAMPLING_RATE_HZ, 25, 0)
    
    # 对齐最终数据
    final_aligned, final_out_aligned,_,_ = alignment(current_signal, pa_output_history[-1], SAMPLING_RATE_HZ, 25, 0)
    
    # 归一化
    initial_norm = initial_aligned / np.max(np.abs(initial_aligned))
    pa_out_initial_norm = initial_out_aligned / np.max(np.abs(initial_out_aligned))
    
    final_norm = final_aligned / np.max(np.abs(final_aligned))
    pa_out_final_norm = final_out_aligned / np.max(np.abs(final_out_aligned))
    
    # 绘制AM/AM特性
    input_mag_initial = np.abs(initial_norm)
    output_mag_initial = np.abs(pa_out_initial_norm)
    input_mag_final = np.abs(final_norm)
    output_mag_final = np.abs(pa_out_final_norm)
    
    print(f"数据对齐完成，初始数据长度: {len(input_mag_initial)}，最终数据长度: {len(input_mag_final)}")
    
       
    axes[1, 0].scatter(input_mag_initial, output_mag_initial, label='初始AM/AM',s=10)
    axes[1, 0].scatter(input_mag_final, output_mag_final,label='ILC AM/AM',s=10)
    axes[1, 0].plot([0, 1], [0, 1], 'k', alpha=0.5, label='理想线性')
    axes[1, 0].set_title('AM/AM特性对比', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('输入幅度')
    axes[1, 0].set_ylabel('输出幅度')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 功率谱密度对比
    plot_PSD_func(pa_out_initial_norm, pa_out_final_norm, SAMPLING_RATE_HZ, axes[1, 1])
    axes[1, 1].set_title('功率谱密度对比 (初始 vs ILC)', fontsize=14, fontweight='bold')
    axes[1, 1].legend(['初始PA输出', 'ILC PA输出'])
    
    # 误差信号演化
    if len(pa_input_history) > 1:
        # 计算每次迭代的误差信号
        error_signals = []
        for i in range(len(pa_input_history)):
            if i < len(pa_output_history) - 1:  # 确保有对应的输出
                pa_ini_norm = initial_norm[:len(pa_input_history[i])]
                pa_out_norm = pa_output_history[i+1] / np.max(np.abs(pa_output_history[i+1]))
                pa_out_norm = pa_out_norm[:len(pa_ini_norm)]  # 确保长度一致
                error = np.abs(pa_ini_norm - pa_out_norm)
                error_signals.append(np.mean(error))
        
        error_iterations = range(1, len(error_signals) + 1)
        axes[1, 2].semilogy(error_iterations, error_signals, 'mo-', linewidth=2, markersize=6)
        axes[1, 2].set_title('误差信号收敛性', fontsize=14, fontweight='bold')
        axes[1, 2].set_xlabel('迭代次数')
        axes[1, 2].set_ylabel('平均误差幅度 (对数)')
        axes[1, 2].grid(True, alpha=0.3)
    else:
        axes[1, 2].text(0.5, 0.5, '需要更多迭代数据', ha='center', va='center', 
                       transform=axes[1, 2].transAxes, fontsize=12)
        axes[1, 2].set_title('误差信号收敛性', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # 性能指标总结
    print("\\n=== ILC性能指标总结 ===")
    print(f"原始信号PAPR: {cal_PAPR(initial_norm):.4f} dB")
    
    for i, pa_output in enumerate(pa_output_history):
        pa_output_norm = pa_output / np.max(np.abs(pa_output))
        print(f"第{i}次PA输出PAPR: {cal_PAPR(pa_output_norm):.4f} dB")
    
    print(f"\\n最终ILC信号PAPR: {cal_PAPR(current_signal / np.max(np.abs(current_signal))):.4f} dB")
    
    # ILC算法收敛性分析
    print("\\n=== ILC收敛性分析 ===")
    if len(nmse_history) > 2:
        # 计算NMSE改善率
        nmse_improvement = nmse_history[-1] - nmse_history[0]
        convergence_rate = nmse_improvement / len(nmse_history)
        print(f"NMSE总改善: {nmse_improvement:.4f} dB")
        print(f"平均收敛速率: {convergence_rate:.4f} dB/次")
        
        # 判断收敛状态
        if len(nmse_history) >= 3:
            last_three_changes = [nmse_history[i] - nmse_history[i-1] for i in range(-2, 0)]
            avg_recent_change = np.mean(last_three_changes)
            if abs(avg_recent_change) < 0.5:  # 阈值可调
                print("算法状态: 已收敛")
            else:
                print("算法状态: 仍在收敛中")
    
    # 最终改善效果统计
    final_acp = aclr_results_history[-1]
    initial_acp = aclr_results_history[0]
    
    print("\\n=== 最终改善效果 ===")
    print(f"邻道改善: 低端 {float(final_acp[2]) - float(initial_acp[2]):.2f} dB, 高端 {float(final_acp[4]) - float(initial_acp[4]):.2f} dB")
    print(f"次邻道改善: 低端 {float(final_acp[6]) - float(initial_acp[6]):.2f} dB, 高端 {float(final_acp[8]) - float(initial_acp[8]):.2f} dB")
       
    print(f"✅ 当前配置：gamma = {ILC_GAMMA}, 迭代{len(aclr_results_history)}次")
    
    print("\\n🎉 基于ILC的自动化数字预失真测试完成！")

else:
    print("已跳过效果对比分析")


In [ ]:
# 传统多项式DPD
import sys
sys.path.append(os.path.abspath("code_ordered"))
import Tradition_function
from Tradition_function import *
reload(Tradition_function)

# 功能控制开关
ENABLE_SMW200A = 1               # 是否启用信号发生器
ENABLE_DPD_ITERATIONS = 1        # 是否执行DPD迭代循环
ENABLE_COMPARISON = 1            # 是否执行效果对比分析
ENABLE_SWEEP = 1                 # 是否执行扫频分析
ENABLE_ACPR = 1                  # 是否执行邻道功率测量
ENABLE_IQ_ANALYSIS = 1           # 是否执行IQ分析
ENABLE_SAVE_IQ = 1               # 是否保存IQ数据到文件
ENABLE_PLOT_PSD = 1              # 是否绘制功率谱密度图
ENABLE_UPLOAD_WAVEFORM = 1       # 是否上传波形文件到信号发生器

# 数字预失真算法参数
model_name_all = ['M_MSA', 'I_MSA', 'MP', 'GMP', 'DDR', 'DVR']
MODEL_SELECTION_MODE = 1                   # 1=测试所有模型，0=只测试指定模型
DPD_MODEL_INDEX = 3                        # 选择GMP模型（当MODEL_SELECTION_MODE=0时使用）
DPD_ITERATIONS = 0                         # DPD迭代次数
mem_depth = 7                              # 记忆深度
K = 5                                      # 非线性阶数
L = 2                                      # GMP模型专用参数
isreversal = 0                             # 1=逆模型，0=正模型

out_data_file_name = "ILC_iter_4.txt"

# ============================
# 载入原始信号数据
# ============================
print("\n载入原始信号数据...")
data_ini_IQ, _, _ = load_data(data_ini_path, data_ini_name)
initial_signal = data_ini_IQ[:, 0] + 1j*data_ini_IQ[:, 1]
print(f"原始信号载入完成，长度: {len(initial_signal)}")

# ============================
# DPD迭代循环处理
# ============================

if ENABLE_DPD_ITERATIONS:
    print("\\n=== 开始DPD迭代循环处理 ===")
    
    # 存储所有模型的结果
    all_models_results = {}
    
    # 确定要测试的模型列表
    if MODEL_SELECTION_MODE == 1:
        models_to_test = model_name_all  # 测试所有模型
        print(f"将测试所有模型: {models_to_test}")
    else:
        models_to_test = [model_name_all[DPD_MODEL_INDEX]]  # 只测试指定模型
        print(f"将测试指定模型: {models_to_test}")

    output_file = os.path.join(data_out_path, out_data_file_name)
        
    if os.path.exists(output_file):
        # input_data = np.loadtxt(input_file)
        output_data = np.loadtxt(output_file)
        
        current_signal = initial_signal[1000:]
        pa_output_complex = output_data[1000:, 0] + 1j*output_data[1000:, 1]
        
        print(f"已加载指定文件，输入信号长度: {len(current_signal)}，输出信号长度: {len(pa_output_complex)}")
    else:
        print(f"警告: 指定文件不存在，使用当前采集的信号")
    
    # 数据对齐
    pa_in_c, pa_out_c, bias, _ = alignment(current_signal, pa_output_complex, SAMPLING_RATE_HZ, 25, 1)
    print(f"数据对齐偏移: {bias}")
    
    # 归一化
    pa_in_norm = pa_in_c / np.max(np.abs(pa_in_c))
    pa_out_norm = pa_out_c / np.max(np.abs(pa_out_c))
    initial_norm = initial_signal / np.max(np.abs(initial_signal))
    
    # 模型选择循环
    for model_idx, model_name in enumerate(models_to_test):
        print(f"\n=== 开始测试模型 {model_idx+1}/{len(models_to_test)}: {model_name} ===")
        
        # 存储当前模型每次迭代的结果
        aclr_results_history = []
        pa_output_history = []
        max_NMSE_history = []
        if model_idx == 0:
            current_signal = initial_signal  # 当前信号发生器要发射的信号
                
        # ============================
        # 步骤1：DPD处理（
        # ============================
        
        # DPD输入信号扩展（基于原始信号）
        o_len = len(initial_norm)
        if model_name == 'GMP':
            orig_complex = np.append(np.append(initial_norm[o_len-mem_depth-L:], initial_norm), 
                                    initial_norm[:L])
            print(f'{model_name}: M:{mem_depth} K:{K} L:{L}')                    
            dpd_path = data_out_path +"ILC_"+ model_name+"_" + str(mem_depth) + "_"+ str(K) + "_"+ str(L) + "_dpd" + ".txt"
            
        else:
            orig_complex = np.append(initial_norm[o_len-mem_depth:], initial_norm)
            print(f'{model_name}: M:{mem_depth} K:{K}')
            dpd_path = data_out_path+"ILC_" + model_name+"_" + str(mem_depth) + "_"+ str(K) + "_dpd" + ".txt"

        
        # 根据模型进行DPD处理
        if model_name == 'M_MSA':
            beta = [k/K for k in range(K+1)]
            if isreversal == 1:
                sam_input, sam_output, sam_xn = generate_data_for_mmsa(pa_out_norm, pa_in_norm, mem_depth)
            else:
                sam_input, sam_output, sam_xn = generate_data_for_mmsa(pa_in_norm, pa_out_norm, mem_depth)
            
            para = sort_and_compute_para_for_mmsa(sam_xn, sam_input, sam_output, K, beta)
            y_fit = compute_u_for_mmsa(sam_xn, sam_input, K, para, beta)
            Max_NMSE = cal_NMSE_np(sam_output, y_fit)
            
            sam_input, _, sam_xn = generate_data_for_mmsa(orig_complex, pa_out_norm, mem_depth)
            dpd_signal = compute_u_for_mmsa(sam_xn, sam_input, K, para, beta)
        
        elif model_name == 'I_MSA':
            beta = [k/K for k in range(K+1)]
            if isreversal == 1:
                sam_input, sam_output = generate_data_for_imsa(pa_out_norm, pa_in_norm, mem_depth)
            else:
                sam_input, sam_output = generate_data_for_imsa(pa_in_norm, pa_out_norm, mem_depth)
            
            para = sort_and_compute_para_for_imsa(sam_input, sam_output, K, beta)
            y_fit, _ = compute_u_for_imsa(sam_input, K, para, beta)
            Max_NMSE = cal_NMSE_np(sam_output, y_fit)
            
            sam_input, _ = generate_data_for_imsa(orig_complex, pa_out_norm, mem_depth)
            dpd_signal, _ = compute_u_for_imsa(sam_input, K, para, beta)
            
        elif model_name == 'MP':
            if isreversal == 1:
                sam_input, sam_output = generate_data_for_mp(pa_out_norm, pa_in_norm, mem_depth, K)
                para, x_fit = compute_u_for_mp(sam_input, sam_output)
                Max_NMSE = cal_NMSE_np(pa_in_norm[mem_depth:], x_fit)
            else:
                sam_input, sam_output = generate_data_for_mp(pa_in_norm, pa_out_norm, mem_depth, K)
                para, x_fit = compute_u_for_mp(sam_input, sam_output)
                Max_NMSE = cal_NMSE_np(pa_out_norm[mem_depth:], x_fit)
            
            x_array, _ = generate_data_for_mp(orig_complex, orig_complex, mem_depth, K)
            dpd_signal = np.dot(x_array, para)
            
        elif model_name == 'GMP':
            if isreversal == 1:
                sam_input, sam_output = generate_data_for_gmp(pa_out_norm, pa_in_norm, mem_depth, K, L)
                para, x_fit = compute_u_for_gmp(sam_input, sam_output)
                Max_NMSE = cal_NMSE_np(pa_in_norm[mem_depth+L:-L], x_fit)
            else:
                sam_input, sam_output = generate_data_for_gmp(pa_in_norm, pa_out_norm, mem_depth, K, L)
                para, y_fit = compute_u_for_gmp(sam_input, sam_output)
                Max_NMSE = cal_NMSE_np(pa_out_norm[mem_depth+L:-L], y_fit)
            
            x_array, _ = generate_data_for_gmp(orig_complex, orig_complex, mem_depth, K, L)
            dpd_signal = np.dot(x_array, para)
            
        elif model_name == 'DDR':
            if isreversal == 1:
                sam_input, sam_output = generate_data_for_ddr(pa_out_norm, pa_in_norm, mem_depth, K)
                para, x_fit = compute_u_for_ddr(sam_input, sam_output)
                Max_NMSE = cal_NMSE_np(pa_in_norm[mem_depth:], x_fit)
            else:
                sam_input, sam_output = generate_data_for_ddr(pa_in_norm, pa_out_norm, mem_depth, K)
                para, x_fit = compute_u_for_ddr(sam_input, sam_output)
                Max_NMSE = cal_NMSE_np(pa_out_norm[mem_depth:], x_fit)
            
            x_array, _ = generate_data_for_ddr(orig_complex, orig_complex, mem_depth, K)
            dpd_signal = np.dot(x_array, para)
            
        elif model_name == 'DVR':  # DVR
            beta = [k/K for k in range(K+1)]
            if isreversal == 1:
                X_base, U = generate_data_for_dvr(pa_out_norm, pa_in_norm, mem_depth, beta)
                para = np.dot(np.linalg.pinv(X_base), U)
                x_fit = np.dot(X_base, para)
                Max_NMSE = cal_NMSE_np(pa_in_norm[mem_depth:], x_fit)
            else:
                X_base, U = generate_data_for_dvr(pa_in_norm, pa_out_norm, mem_depth, beta)
                para = np.dot(np.linalg.pinv(X_base), U)
                x_fit = np.dot(X_base, para)
                Max_NMSE = cal_NMSE_np(pa_out_norm[mem_depth:], x_fit)
            
            dpd_signal = compute_u_for_dvr(orig_complex, mem_depth, beta, para)
        
        max_NMSE_history.append(Max_NMSE)

        # 输出DPD处理结果
        print(f"参数动态范围: {20 * np.log10(np.abs(np.max(para)-np.min(para))):.4f}dB")
        print(f"NMSE: {np.array(Max_NMSE):.4f}dB")
        print(f"PAPR_in: {cal_PAPR(pa_in_norm):.4f}dB, PAPR_out: {cal_PAPR(pa_out_norm):.4f}dB, PAPR_dpd: {cal_PAPR(dpd_signal):.4f}dB")
        
        # 保存DPD信号
        # dpd_file_name = f"{model_name}_{mem_depth}_{K}_{L}_dpd_{iteration+1}.txt"
        # dpd_path = os.path.join(data_out_path, dpd_file_name)
        save_txt_func(dpd_signal, dpd_path)
        print(f"DPD信号已保存到: {dpd_path}")
        
        # 更新当前信号输入
        current_signal = dpd_signal
        

        # ============================
        # 步骤2：配置信号发生器发射当前信号
        # ============================
        
        # 创建并上传DPD波形文件
        RsSmw.assert_minimum_version('5.0.44')
        smw = RsSmw(f'TCPIP0::{SMW200A_IP}::inst0::INSTR')
        rm = pyvisa.ResourceManager()
        usr_SMW200 = rm.open_resource(f"TCPIP0::{SMW200A_IP}::inst0::INSTR")
        usr_SMW200.timeout = 30000
        
        usr_SMW200.write('OUTP:STAT OFF')
        usr_SMW200.write(f'MMEM:MDIR "{INST_WV_DIR}"')
        
        # # 保存当前信号到文件
        # current_signal_file = os.path.join(LOCAL_DIR, f'current_signal_iter_{iteration}.txt')
        # current_signal_data = np.column_stack([current_signal.real, current_signal.imag])
        # np.savetxt(current_signal_file, current_signal_data, fmt='%.8f')
        if model_name == 'GMP':
            dpd_path_wv = data_out_path+"ILC_" + model_name+"_" + str(mem_depth) + "_"+ str(K) + "_"+ str(L) + "_dpd" + ".wv"
            INST_WV_FILE = f"{INST_WV_DIR}/ILC_{model_name}_{mem_depth}_{K}_{L}_dpd.wv"
        else:
            dpd_path_wv = data_out_path+"ILC_" + model_name+"_" + str(mem_depth) + "_"+ str(K) + "_dpd" + ".wv"
            INST_WV_FILE = f"{INST_WV_DIR}/ILC_{model_name}_{mem_depth}_{K}_dpd.wv"
        
        # 创建波形文件
        LOCAL_WV_FILE = os.path.join(LOCAL_DIR, dpd_path_wv)
        # INST_WV_FILE = f"{INST_WV_DIR}/dpd_signal_iter_{iteration}.wv"
        
        smw.arb_files.create_waveform_file_from_samples(
            current_signal.real, current_signal.imag, LOCAL_WV_FILE,
            clock_freq=SAMPLING_RATE_HZ, auto_scale=True,
            additional_tags=['MARKER LIST 1: 0:1;10:0'],
            comment=f'DPD iteration {iteration}'
        )
        
        # 上传波形文件
        with open(LOCAL_WV_FILE, 'rb') as file:
            file_content = file.read()
        
        data_len = len(file_content)
        len_str = str(data_len)
        header = f'#{len(len_str)}{len_str}'
        
        usr_SMW200.write_raw(f'MMEM:DATA "{INST_WV_FILE}",'.encode() + header.encode() + file_content)
        usr_SMW200.write(f'SOURce:BB:ARB:WAV:SELect "{INST_WV_FILE}"')
        usr_SMW200.write('SOURce:BB:ARB:STATe ON')
        usr_SMW200.write('OUTP:STAT ON')
    
        usr_SMW200.close()
        rm.close()
        
        # ============================
        # 步骤2：频谱仪测量PA输出
        # ============================
        print("频谱仪测量PA输出...")
        rm_ceyear = pyvisa.ResourceManager()
        usr_ceyear = rm_ceyear.open_resource(f"TCPIP0::{CEYEAR_IP}::5025::SOCKET")
        usr_ceyear.set_visa_attribute(pyvisa.constants.VI_ATTR_TERMCHAR_EN, pyvisa.constants.VI_TRUE)
        usr_ceyear.set_visa_attribute(pyvisa.constants.VI_ATTR_TERMCHAR, ord('\n'))
        usr_ceyear.set_visa_attribute(pyvisa.constants.VI_ATTR_TMO_VALUE, 10000)
        
        # ACLR测量
        if ENABLE_ACPR:
            usr_ceyear.write(":CONFigure:ACPower")
            time.sleep(1)
            usr_ceyear.write(f":BAND {RBW_ACLR}")
            usr_ceyear.write(f":BAND:VID {VBW_ACLR}")
            usr_ceyear.write(":RAD:STAN NR5GFR1B100M")
            usr_ceyear.write(":TRAC1:ACP:TYPE AVERage")
            time.sleep(2)
            usr_ceyear.write(":FETC:ACP?")
            acp_results = usr_ceyear.read().strip().split(',')
            aclr_results_history.append(acp_results)
            
            print(f"{model_name}模型ACLR结果:")
            print(f"  载波功率: {acp_results[0]} dBm")
            print(f"  邻道: 低相对: {acp_results[2]} dBc, 高相对: {acp_results[4]} dBc")
            print(f"  次邻道: 低相对: {acp_results[6]} dBc, 高相对: {acp_results[8]} dBc")
        
        
        # IQ数据采集
        if ENABLE_IQ_ANALYSIS:
            usr_ceyear.write(":INST BASIC")
            usr_ceyear.write(f":FREQ:CENT {CENTER_FREQ}")
            usr_ceyear.write("ROSC:SOUR E10")
            usr_ceyear.write("TRIG:SOUR EXT1")
            usr_ceyear.write(f'TRAC:IQ:SRAT {SAMPLING_RATE}')
            # usr_ceyear.write(f'SWE:TIME {MEASUREMENT_TIME}')
            usr_ceyear.write(f'TRAC:IQ:RLEN {IQ_RECORD_LENGTH}')
            usr_ceyear.write(':ADJ:LEV')
            time.sleep(5)
            
            usr_ceyear.write('INIT:IMM')
            usr_ceyear.clear()
            time.sleep(1)
            
            # 读取IQ数据
            usr_ceyear.timeout = 30000
            usr_ceyear.write(":FORM:DATA REAL,32")
            usr_ceyear.write(":TRAC:IQ:DATA?")
            
            usr_ceyear.set_visa_attribute(pyvisa.constants.VI_ATTR_TERMCHAR_EN, pyvisa.constants.VI_FALSE)
            
            # 读取二进制块头
            header = usr_ceyear.read_bytes(2)
            digit_count = header[1] - 48
            length_str = usr_ceyear.read_bytes(digit_count).decode('ascii')
            data_length = int(length_str)
            
            # 分块读取数据
            raw_data = bytearray()
            chunk_size = 1024 * 1024
            remaining = data_length
            
            while remaining > 0:
                chunk = min(remaining, chunk_size)
                raw_data.extend(usr_ceyear.read_bytes(chunk))
                remaining -= chunk
            
            usr_ceyear.read_bytes(1)
            
            # 转换为浮点数数组
            float_count = len(raw_data) // 4
            iq_data_current = struct.unpack(f'{float_count}f', raw_data)
            usr_ceyear.set_visa_attribute(pyvisa.constants.VI_ATTR_TERMCHAR_EN, pyvisa.constants.VI_TRUE)
            
            # 转换为复数并保存
            pa_output_current = np.array([[iq_data_current[i], iq_data_current[i+1]] 
                                            for i in range(0, len(iq_data_current), 2)])
            pa_output_complex = pa_output_current[:, 0] + 1j*pa_output_current[:, 1]
            pa_output_history.append(pa_output_complex)
            
            # 保存PA输出数据
            if ENABLE_SAVE_IQ:
                if model_name == 'GMP':
                    pa_output_file = data_out_path +"ILC_" + model_name+"_" + str(mem_depth) + "_"+ str(K) + "_"+ str(L) + "_dpd_out" + ".txt"
                    
                else:
                    pa_output_file = data_out_path +"ILC_" + model_name+"_" + str(mem_depth) + "_"+ str(K) + "_dpd_out" + ".txt"
                    
                np.savetxt(pa_output_file, pa_output_current, fmt='%.8f')
                print(f"{model_name}的PA输出数据已保存，共{len(pa_output_complex)}个点")
            else:
                print(f"{model_name}的PA输出数据已采集，共{len(pa_output_complex)}个点（未保存）")
            
            usr_ceyear.write('INIT:CONT ON')
            # usr_ceyear.close()
            # rm_ceyear.close()
        

        # 配置扫频分析设置
        if ENABLE_SWEEP:
            print("执行扫频分析...")
            usr_ceyear.write(":INST SA")                     # 切换到频谱分析模式
            usr_ceyear.write(f":FREQ:CENT {CENTER_FREQ}")    # 设置中心频率
            usr_ceyear.write(f":FREQ:SPAN {SWEEP_RANGE}")    # 设置扫频范围
            
            # 设置RBW和VBW用于占用带宽测量
            # usr_ceyear.write(f":BAND {RBW_OCCUPIED}")    # 设置分辨带宽
            # usr_ceyear.write(f":BAND:VID {VBW_OCCUPIED}")    # 设置视频带宽
            
            time.sleep(1)
            
            # 配置6条曲线的循环显示逻辑
            # 首先确保所有曲线都是关闭的，只在需要时打开
            if model_idx == 0:
                for i in range(2, 7):
                    usr_ceyear.write(f":TRAC{i}:DISP OFF")
            
            # 确定当前模型的曲线显示范围
            current_trace = model_idx % 5 + 2  # 当前模型的起始曲线位置                
                            
            # 设置当前曲线进行测量
            usr_ceyear.write(f":TRAC{current_trace}:DISP ON")
            usr_ceyear.write(f":TRAC{current_trace}:TYPE AVER")  # 当前曲线设置为平均模式
            usr_ceyear.write(f":TRAC{current_trace}:UPD ON")     # 开启当前曲线更新
            # print(f"曲线{current_trace}正在测量{model_name}模型迭代{iteration}")
            time.sleep(5)  # 等待当前曲线平均完成
            usr_ceyear.write(f":TRAC{current_trace}:UPD OFF")     # 开启当前曲线更新                  

        usr_ceyear.close()
        rm_ceyear.close()

        # 保存当前模型的结果
        all_models_results[model_name] = {
            'aclr_history': aclr_results_history.copy(),
            'pa_output_history': pa_output_history.copy(),
            'max_NMSE_history': max_NMSE_history.copy(),
            'final_signal': current_signal.copy()
        }
        print(f"\\n模型 {model_name} 测试完成！")
        
    print("\\n所有模型DPD迭代循环完成！")    

    
    # 打印所有模型的结果摘要
    print("\\n=== 所有模型结果摘要 ===")
    for model_name, results in all_models_results.items():
        if results['aclr_history']:
            final_aclr = results['aclr_history'][-1]
            print(f"{model_name}: 最终ACLR - 邻道低: {final_aclr[2]} dBc, 邻道高: {final_aclr[4]} dBc")

else:
    print("已跳过DPD迭代循环")
